# Avances III – Iteración A6: *transformer* para el sentimiento de los comentarios
**Proyecto Estadísticas de YouTube · Procesamiento de Lenguaje Natural · Universidad de Cundinamarca**

Este notebook se ejecuta en **Google Colab con GPU** (Entorno de ejecución → Cambiar tipo de entorno → GPU T4).

1. Ejecute todas las celdas (Entorno de ejecución → Ejecutar todas).
2. Cuando se solicite, suba `videos-stats.csv` y `comments.csv`.
3. Al final se descarga `oof_transformer.csv` y `resultados_transformer.json`.

Se evalúan dos variantes con **los mismos 5 folds** que el resto de iteraciones:
- **A6a – Zero-shot:** `cardiffnlp/twitter-roberta-base-sentiment-latest` (RoBERTa entrenado con ~124 M de tuits) sin reentrenar.
- **A6b – Ajuste fino (fine-tuning):** el mismo modelo reentrenado con las etiquetas del dataset en cada fold.

In [ ]:
!pip -q install emoji transformers accelerate

In [ ]:
import torch, numpy as np, pandas as pd, json, time
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
from google.colab import files
up = files.upload()  # seleccione videos-stats.csv y comments.csv

In [ ]:
# Funciones comunes (idénticas a src/comun.py del repositorio)
"""Funciones comunes de carga y limpieza (idénticas a las de Avances II) para las iteraciones de Avances III."""
import re
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, KFold

SEED = 42


def cargar(vid_path, com_path):
    v = pd.read_csv(vid_path)
    c = pd.read_csv(com_path)
    v = v.drop(columns=[x for x in ["Unnamed: 0"] if x in v]); c = c.drop(columns=[x for x in ["Unnamed: 0"] if x in c])
    v = v.drop_duplicates("Video ID").dropna(subset=["Views"])
    for col in ["Likes", "Comments"]:
        v.loc[v[col] < 0, col] = np.nan
    v = v[v.Views > 0].copy()
    v["Published At"] = pd.to_datetime(v["Published At"], errors="coerce")
    v["age_days"] = (v["Published At"].max() - v["Published At"]).dt.days.clip(lower=1)
    c = c.dropna(subset=["Comment", "Sentiment"]).drop_duplicates(["Video ID", "Comment"])
    c["Sentiment"] = c["Sentiment"].astype(int)
    c = c[c.Sentiment.isin([0, 1, 2]) & c["Video ID"].isin(v["Video ID"])].reset_index(drop=True)
    c = c.merge(v[["Video ID", "Keyword"]], on="Video ID", how="left")
    return v.reset_index(drop=True), c


def limpiar_basico(s):
    """Limpieza de Avances II."""
    s = str(s).lower()
    s = re.sub(r"http\S+|www\.\S+", " ", s)
    s = re.sub(r"[@#]\w+", " ", s)
    s = re.sub(r"[^a-záéíóúñü0-9\s!?']", " ", s)
    return re.sub(r"\s+", " ", s).strip()


NEG = {"not", "no", "never", "nothing", "nobody", "none", "nor", "cannot", "dont", "doesnt", "didnt", "isnt", "wasnt",
       "arent", "werent", "wont", "wouldnt", "shouldnt", "couldnt", "cant", "aint", "hardly", "without"}


def limpiar_mejorado(s):
    """Iteración A1: emojis a texto, alargamientos, contracciones negativas y marcado de negación."""
    import emoji
    s = emoji.demojize(str(s), delimiters=(" EMO_", " "))
    s = s.lower()
    s = re.sub(r"http\S+|www\.\S+", " URL ", s)
    s = re.sub(r"@\w+", " USER ", s)
    s = re.sub(r"n't\b", " not", s).replace("’", "'")
    s = re.sub(r"(.)\1{2,}", r"\1\1", s)                  # sooooo -> soo
    s = re.sub(r"!{2,}", " MULTIEXCL ", s); s = re.sub(r"\?{2,}", " MULTIQ ", s)
    s = re.sub(r"[^a-z0-9_\s!?']", " ", s)
    toks, out, neg = s.split(), [], 0
    for t in toks:
        if t in {"!", "?", ".", ","} or t.endswith(("!", "?")):
            neg = 0
        out.append(("NEG_" + t) if neg > 0 and t not in NEG else t)
        if t in NEG: neg = 3                                   # ventana de 3 tokens
        elif neg > 0: neg -= 1
    return " ".join(out)


def folds_a(y):
    return list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))


def folds_b(n):
    return list(KFold(5, shuffle=True, random_state=SEED).split(np.zeros(n)))

In [ ]:
v, c = cargar('videos-stats.csv', 'comments.csv')
y = c.Sentiment.values; F = folds_a(y)
print(len(c), np.bincount(y))
assert len(c) == 18139 and list(np.bincount(y)) == [2328, 4590, 11221], 'Los datos no coinciden con los del informe'

## A6a – Zero-shot

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score, accuracy_score
MODEL = 'cardiffnlp/twitter-roberta-base-sentiment-latest'
tok = AutoTokenizer.from_pretrained(MODEL)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
def preprocess(t):  # normalización recomendada por los autores del modelo
    return ' '.join('@user' if w.startswith('@') and len(w) > 1 else 'http' if w.startswith('http') else w for w in str(t).split())
texts = [preprocess(t) for t in c.Comment.astype(str)]
@torch.no_grad()
def predecir(model, idx, bs=128):
    model.eval(); out = []
    for i in range(0, len(idx), bs):
        b = tok([texts[j] for j in idx[i:i+bs]], padding=True, truncation=True, max_length=128, return_tensors='pt').to(dev)
        with torch.autocast(dev, enabled=dev == 'cuda'):
            out.append(torch.softmax(model(**b).logits.float(), -1).cpu().numpy())
    return np.vstack(out)
m0 = AutoModelForSequenceClassification.from_pretrained(MODEL).to(dev)
print(m0.config.id2label)   # 0 negative, 1 neutral, 2 positive (mismo orden que el dataset)
t0 = time.time(); P0 = predecir(m0, np.arange(len(texts))); z = P0.argmax(1)
zs = {'f1_macro_folds': [f1_score(y[te], z[te], average='macro') for tr, te in F], 'acc': accuracy_score(y, z), 'sec': time.time() - t0}
print('Zero-shot F1-macro', np.mean(zs['f1_macro_folds']), 'exactitud', zs['acc'])
del m0; torch.cuda.empty_cache()

## A6b – Ajuste fino con los mismos 5 folds

In [ ]:
from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup
EPOCHS, BS, LR = 2, 32, 2e-5
w = torch.tensor(len(y) / (3 * np.bincount(y)), dtype=torch.float).to(dev)   # pesos de clase balanceados
P = np.zeros((len(y), 3)); ft_f1 = []; t0 = time.time()
for k, (tr, te) in enumerate(F):
    torch.manual_seed(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * int(np.ceil(len(tr) / BS)); sch = get_linear_schedule_with_warmup(opt, int(.06 * steps), steps)
    scaler = torch.cuda.amp.GradScaler(enabled=dev == 'cuda'); lossf = torch.nn.CrossEntropyLoss(weight=w)
    for ep in range(EPOCHS):
        model.train(); perm = np.random.default_rng(SEED + ep).permutation(tr)
        for i in range(0, len(perm), BS):
            bi = perm[i:i+BS]
            b = tok([texts[j] for j in bi], padding=True, truncation=True, max_length=128, return_tensors='pt').to(dev)
            with torch.autocast(dev, enabled=dev == 'cuda'):
                loss = lossf(model(**b).logits.float(), torch.tensor(y[bi]).to(dev))
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step()
    P[te] = predecir(model, te); f = f1_score(y[te], P[te].argmax(1), average='macro'); ft_f1.append(f)
    print(f'fold {k+1}: F1-macro {f:.4f}  ({(time.time()-t0)/60:.1f} min)')
    del model; torch.cuda.empty_cache()
print('Ajuste fino F1-macro', np.mean(ft_f1), '±', np.std(ft_f1))

## Exportar resultados

In [ ]:
pd.DataFrame({'Video ID': c['Video ID'], 'y': y, 'zs_p0': P0[:,0], 'zs_p1': P0[:,1], 'zs_p2': P0[:,2],
              'ft_p0': P[:,0], 'ft_p1': P[:,1], 'ft_p2': P[:,2]}).to_csv('oof_transformer.csv', index=False)
res = {'modelo': MODEL, 'zero_shot': {**zs, 'f1_macro': float(np.mean(zs['f1_macro_folds']))},
       'ajuste_fino': {'f1_macro_folds': [float(x) for x in ft_f1], 'f1_macro': float(np.mean(ft_f1)), 'f1_sd': float(np.std(ft_f1)),
                       'acc': float(accuracy_score(y, P.argmax(1))), 'epochs': EPOCHS, 'batch': BS, 'lr': LR, 'min': (time.time() - t0) / 60},
       'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}
json.dump(res, open('resultados_transformer.json', 'w'), indent=1, default=float); print(res)
files.download('oof_transformer.csv'); files.download('resultados_transformer.json')